In [1]:
import pandas as pd

ETF_FILE = "../data/raw/ETF.xlsx"
RF_FILE = "../data/raw/dgs1mo.csv"

xls = pd.ExcelFile(ETF_FILE)
print("Feuilles du fichier :", xls.sheet_names)

Feuilles du fichier : ['Price', 'Total return index', 'interest rate ']


In [2]:
pd.set_option("display.max_columns", 20)

for sheet in ["Price", "Total return index"]:
    raw = pd.read_excel(ETF_FILE, sheet_name=sheet, header=None)
    print(f"--- {sheet} : {raw.shape[0]} lignes, {raw.shape[1]} colonnes ---")
    display(raw.head(6))
    display(raw.tail(3))

--- Price : 5179 lignes, 9 colonnes ---


,0,1,2,3,4,5,6,7,8
0,Name,ISHARES MSCI EMERGING MARKETS ETF,ISHARES MSCI EAFE,STATE STREET SPDR S&P 500 ETF TRUST,ISHARES 7-10 YR.TRSY.BD.,ISHARES IBOXX $ INV GRADE CORPORATE BOND ETF,ISHARES 20+ YEAR TREASURY BOND ETF,VANGUARD REAL ESTATE ETF,SPDR GOLD SHARES
1,2007-01-01 00:00:00,38.0566,73.22,141.62,82.44,106.68,88.43,77,63.21
2,2007-01-02 00:00:00,38.0566,73.22,141.62,82.44,106.68,88.43,77,63.21
3,2007-01-03 00:00:00,38.38,73.51,141.37,82.68,106.45,89.07,77.12,62.28
4,2007-01-04 00:00:00,37.85,73.32,141.67,82.95,107.25,89.6,77.22,61.65
5,2007-01-05 00:00:00,36.7433,72.2,140.54,82.71,107.19,89.22,75.79,60.17


,0,1,2,3,4,5,6,7,8
5176,2026-11-02 00:00:00,61.11,103.7,752.1001,91.5,103.71,81.73,95.06,361.6499
5177,2026-11-03 00:00:00,61.1,103.08,754.3501,91.16,103.72,81.11,95.01,359.26
5178,2026-11-04 00:00:00,61.62,105.72,766.8101,91.66,104.5,81.76,98.03,372.49


--- Total return index : 5179 lignes, 9 colonnes ---


,0,1,2,3,4,5,6,7,8
0,Name,STATE STREET SPDR S&P 500 ETF TRUST - TOT RETU...,ISHARES MSCI EAFE - TOT RETURN IND,ISHARES MSCI EMERGING MARKETS ETF - TOT RETURN...,ISHARES 7-10 YR.TRSY.BD. - TOT RETURN IND,ISHARES IBOXX $ INV GRADE CORPORATE BOND ETF -...,ISHARES 20+ YEAR TREASURY BOND ETF - TOT RETUR...,VANGUARD REAL ESTATE ETF - TOT RETURN IND,SPDR GOLD SHARES - TOT RETURN IND
1,2007-01-01 00:00:00,410.44,190.23,358.35,118.85,129.76,132.61,175.65,142.43
2,2007-01-02 00:00:00,410.44,190.23,358.35,118.85,129.76,132.61,175.65,142.43
3,2007-01-03 00:00:00,409.72,190.99,361.39,119.2,129.48,133.57,175.93,140.33
4,2007-01-04 00:00:00,410.59,190.49,356.4,119.59,130.45,134.37,176.15,138.91
5,2007-01-05 00:00:00,407.31,187.58,345.98,119.24,130.38,133.8,172.89,135.58


,0,1,2,3,4,5,6,7,8
5176,2026-11-02 00:00:00,3126.42,482.12,858.94,219.81,277.72,229.5,490.59,814.89
5177,2026-11-03 00:00:00,3135.78,479.24,858.8,218.99,277.75,227.75,490.33,809.51
5178,2026-11-04 00:00:00,3187.57,491.51,866.11,220.19,279.84,229.58,505.92,839.32


In [3]:
TICKER_KEYWORDS = {
    "SPY": "S&P 500",
    "EFA": "EAFE",
    "EEM": "EMERGING",
    "IEF": "7-10",
    "LQD": "INV GRADE",
    "TLT": "20+",
    "VNQ": "REAL ESTATE",
    "GLD": "GOLD",
}


def read_datastream_sheet(sheet):
    df = pd.read_excel(ETF_FILE, sheet_name=sheet, header=0)
    df = df.rename(columns={df.columns[0]: "Date"})

    # Contrôler les dates
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    print(f"\n--- {sheet} ---")
    print("Dates invalides :", df["Date"].isna().sum())
    print("Dates dupliquées :", df["Date"].dropna().duplicated().sum())
    df = df.dropna(subset=["Date"])

    # Identifier les ETF par leur nom
    mapping = {}
    for col in df.columns[1:]:
        matches = [t for t, kw in TICKER_KEYWORDS.items() if kw in str(col).upper()]
        assert len(matches) == 1, f"Colonne ambiguë ou non reconnue : {col}"
        mapping[col] = matches[0]
    assert len(mapping) == 8
    assert set(mapping.values()) == set(TICKER_KEYWORDS)
    df = df.rename(columns=mapping)

    # Contrôler les valeurs non numériques
    for ticker in TICKER_KEYWORDS:
        numeric = pd.to_numeric(df[ticker], errors="coerce")
        invalid = df[ticker].notna() & numeric.isna()
        print(f"{ticker} : {invalid.sum()} valeurs non numériques")
        df[ticker] = numeric

    return df.set_index("Date").sort_index()[list(TICKER_KEYWORDS)]


price = read_datastream_sheet("Price")
ri = read_datastream_sheet("Total return index")

print()
for name, df in [("Prix P", price), ("RI", ri)]:
    print(f"{name} : {df.shape[0]} dates, du {df.index.min().date()} au {df.index.max().date()}")
    print("  Colonnes :", list(df.columns))
print("Mêmes dates dans les deux feuilles :", price.index.equals(ri.index))


--- Price ---
Dates invalides : 0
Dates dupliquées : 0
SPY : 0 valeurs non numériques
EFA : 0 valeurs non numériques
EEM : 0 valeurs non numériques
IEF : 0 valeurs non numériques
LQD : 0 valeurs non numériques
TLT : 0 valeurs non numériques
VNQ : 0 valeurs non numériques
GLD : 0 valeurs non numériques

--- Total return index ---
Dates invalides : 0
Dates dupliquées : 0
SPY : 0 valeurs non numériques
EFA : 0 valeurs non numériques
EEM : 0 valeurs non numériques
IEF : 0 valeurs non numériques
LQD : 0 valeurs non numériques
TLT : 0 valeurs non numériques
VNQ : 0 valeurs non numériques
GLD : 0 valeurs non numériques

Prix P : 5178 dates, du 2007-01-01 au 2026-11-04
  Colonnes : ['SPY', 'EFA', 'EEM', 'IEF', 'LQD', 'TLT', 'VNQ', 'GLD']
RI : 5178 dates, du 2007-01-01 au 2026-11-04
  Colonnes : ['SPY', 'EFA', 'EEM', 'IEF', 'LQD', 'TLT', 'VNQ', 'GLD']
Mêmes dates dans les deux feuilles : True


In [4]:
import pandas_market_calendars as mcal

# ---------------------------------------------------------------------------
# Data quality report: one row per check
# ---------------------------------------------------------------------------
report = []

def add_check(check, ticker, dates, detected, kept=0, excluded=0, corrected=0, justification=""):
    report.append({
        "Check": check,
        "Ticker": ticker,
        "Dates": ", ".join(str(d.date()) for d in dates[:10]) + (" ..." if len(dates) > 10 else ""),
        "Detected": detected,
        "Kept": kept,
        "Excluded": excluded,
        "Corrected": corrected,
        "Justification": justification,
    })

# ---------------------------------------------------------------------------
# 1. NYSE calendar
# ---------------------------------------------------------------------------
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=ri.index.min(), end_date=ri.index.max())
sessions = pd.DatetimeIndex(schedule.index).normalize()

file_dates = ri.index.normalize()
non_sessions = file_dates.difference(sessions)      # in the file, market closed
missing_sessions = sessions.difference(file_dates)  # market open, missing from the file

prev = ri.shift(1).loc[non_sessions]
copies = (ri.loc[non_sessions] == prev).all(axis=1)
n_first = int(len(non_sessions) > 0 and non_sessions[0] == ri.index.min())

print(f"NYSE sessions: {len(sessions)}")
print(f"Non-NYSE dates in file: {len(non_sessions)} (exact copies of previous day: {copies.sum()})")
print(f"Missing NYSE sessions: {len(missing_sessions)}")

add_check("Non-NYSE date", "All", non_sessions, len(non_sessions), excluded=len(non_sessions),
          justification=(f"NYSE closed; {copies.sum()}/{len(non_sessions)} dates match the previous "
                         f"observation; {n_first} first date(s) without predecessor"))
add_check("Missing NYSE session", "All", missing_sessions, len(missing_sessions),
          justification="No case" if len(missing_sessions) == 0 else "Common date excluded")

# Keep NYSE sessions only (raw data stays untouched)
price_clean = price.loc[price.index.isin(sessions)].copy()
ri_clean = ri.loc[ri.index.isin(sessions)].copy()

assert price_clean.index.equals(ri_clean.index), "Price and RI dates are misaligned"
assert ri_clean.index.equals(sessions), (
    "Some NYSE sessions are missing: apply the V1.4 rule (exclude the common date, flag irregular returns)"
)
print(f"After filtering: {len(ri_clean)} sessions, {ri_clean.index.min().date()} to {ri_clean.index.max().date()}")

# ---------------------------------------------------------------------------
# 2. Missing values, daily returns (no filling)
# ---------------------------------------------------------------------------
assert not ri_clean.isna().any().any(), "Missing values in RI: review needed"
assert not price_clean.isna().any().any(), "Missing values in Price: review needed"

r_ri = ri_clean.pct_change(fill_method=None)
r_p = price_clean.pct_change(fill_method=None)

# ---------------------------------------------------------------------------
# 3. Value checks: non-positive values, extreme returns, stale series
# ---------------------------------------------------------------------------
def zero_runs(s, min_len):
    """Runs of at least min_len consecutive zero returns: (start date, length)."""
    z = s.eq(0)
    block = (z != z.shift()).cumsum()
    return [(grp.index[0], len(grp)) for _, grp in s[z].groupby(block[z]) if len(grp) >= min_len]

EXTREME = 0.15
STALE_LEN = 5

for t in ri_clean.columns:
    nonpos = ri_clean.index[(ri_clean[t] <= 0) | (price_clean[t] <= 0)]
    add_check("Non-positive price or RI", t, nonpos, len(nonpos),
              justification="No case" if len(nonpos) == 0 else "Flagged for review")

    ext = r_ri[t][r_ri[t].abs() > EXTREME]
    add_check("Daily return > 15%", t, ext.index, len(ext),
              justification="No case" if len(ext) == 0 else "Flagged for review; no automatic exclusion")

    runs = zero_runs(r_ri[t], STALE_LEN)
    add_check(f"Stale series ({STALE_LEN}+ days)", t, [d for d, _ in runs], len(runs),
              justification="No case" if not runs else "Flagged for review; no automatic exclusion")

# ---------------------------------------------------------------------------
# 4. Review of extreme returns
# ---------------------------------------------------------------------------
flags = []
for t in ri_clean.columns:
    for d, v in r_ri[t][r_ri[t].abs() > EXTREME].items():
        flags.append({
            "Date": d.date(),
            "Ticker": t,
            "RI return": f"{v:+.1%}",
            "Price return": f"{r_p.loc[d, t]:+.1%}",
            "SPY same day": f"{r_ri.loc[d, 'SPY']:+.1%}",
            "RI return next day": f"{r_ri[t].shift(-1).loc[d]:+.1%}",
        })
if flags:
    display(pd.DataFrame(flags).sort_values("Date"))

# Decision after review: market moves consistent with known crises, kept
report_df = pd.DataFrame(report)
mask = (report_df["Check"] == "Daily return > 15%") & (report_df["Detected"] > 0)
report_df.loc[mask, "Kept"] = report_df.loc[mask, "Detected"]
report_df.loc[mask, "Justification"] = (
    "Kept: market moves consistent with the 2008-2009 and 2020 crises; "
    "Price and RI returns agree at displayed precision; "
    "SPY moves in the same direction; no obvious data error"
)
report = report_df.to_dict("records")

display(report_df[["Check", "Ticker", "Detected", "Kept", "Excluded", "Justification"]])

NYSE sessions: 4992
Non-NYSE dates in file: 186 (exact copies of previous day: 185)
Missing NYSE sessions: 0
After filtering: 4992 sessions, 2007-01-03 to 2026-11-04


,Date,Ticker,RI return,Price return,SPY same day,RI return next day
0,2008-10-13,EFA,+15.9%,+15.9%,+14.5%,-2.1%
1,2008-10-13,EEM,+22.8%,+22.8%,+14.5%,-5.0%
2,2008-10-15,EEM,-16.2%,-16.2%,-9.8%,+4.9%
3,2008-10-28,EEM,+25.6%,+25.6%,+11.7%,-6.8%
4,2008-10-28,VNQ,+15.6%,+15.6%,+11.7%,-3.5%
5,2008-11-24,VNQ,+17.0%,+17.0%,+6.9%,+1.6%
6,2008-12-01,VNQ,-19.5%,-19.5%,-8.9%,+13.3%
7,2009-03-23,VNQ,+16.2%,+16.2%,+7.2%,-7.7%
8,2020-03-16,VNQ,-17.7%,-17.7%,-10.9%,+5.8%


,Check,Ticker,Detected,Kept,Excluded,Justification
0,Non-NYSE date,All,186,0,186,NYSE closed; 185/186 dates match the previous ...
1,Missing NYSE session,All,0,0,0,No case
2,Non-positive price or RI,SPY,0,0,0,No case
3,Daily return > 15%,SPY,0,0,0,No case
4,Stale series (5+ days),SPY,0,0,0,No case
5,Non-positive price or RI,EFA,0,0,0,No case
6,Daily return > 15%,EFA,1,1,0,Kept: market moves consistent with the 2008-20...
7,Stale series (5+ days),EFA,0,0,0,No case
8,Non-positive price or RI,EEM,0,0,0,No case
9,Daily return > 15%,EEM,3,3,0,Kept: market moves consistent with the 2008-20...


In [5]:
from pathlib import Path
REPORTS = Path("../data/reports")
gap = (r_ri - r_p).dropna()   # daily gap: RI return minus price return

# Annual performance gap RI - price, full years only
years = r_ri.index.year
annual_ri = (1 + r_ri).groupby(years).prod() - 1
annual_p = (1 + r_p).groupby(years).prod() - 1
annual_gap = (annual_ri - annual_p).iloc[1:-1]   # drop 2007 (start) and 2026 (incomplete)

summary = pd.DataFrame({
    "Mean annual gap RI - Price": annual_gap.mean().map("{:.2%}".format),
    "Days with gap > +0.1%": (gap > 0.001).sum(),
    "Days with gap < -0.1%": (gap < -0.001).sum(),
    "Largest daily gap": gap.abs().max().map("{:.3%}".format),
})
display(summary)

for t in ri_clean.columns:
    neg = gap.index[gap[t] < -0.001]
    add_check("Negative RI - Price gap (> 0.1 pp)", t, neg, len(neg),
              justification="No case" if len(neg) == 0 else "Flagged for review")


# ---------------------------------------------------------------------------
# Daily returns with a regularity flag
# ---------------------------------------------------------------------------
# A return is regular if the two dates are consecutive NYSE sessions
pos = sessions.get_indexer(r_ri.index)
regular = pd.Series(pos, index=r_ri.index).diff().eq(1)

daily_returns = r_ri.copy()
daily_returns["regular"] = regular
daily_returns = daily_returns.iloc[1:]   # first date has no previous observation

print(f"Daily returns: {len(daily_returns)}, irregular: {(~daily_returns['regular']).sum()}")

# ---------------------------------------------------------------------------
# Monthly returns on a common NYSE month-end
# ---------------------------------------------------------------------------
last_data_date = ri_clean.index.max()
last_month_end = last_data_date + pd.offsets.MonthEnd(0)
full_sessions = pd.DatetimeIndex(
    nyse.schedule(start_date=ri_clean.index.min(), end_date=last_month_end).index
).normalize()
month_ends = full_sessions.to_series().groupby(full_sessions.to_period("M")).max()

unavailable = month_ends[~month_ends.isin(ri_clean.index)]
incomplete_final = unavailable[unavailable > last_data_date]    # month not finished yet
missing_month_end = unavailable[unavailable <= last_data_date]  # genuine data gap

print(f"Month-ends: {len(month_ends)}")
print("Incomplete final month:", [str(p) for p in incomplete_final.index])
print("Missing month-ends (data gap):", [str(p) for p in missing_month_end.index])

# Keep the full monthly calendar: a missing month-end stays visible as NaN
ri_month_end = ri_clean.reindex(month_ends.values)
ri_month_end.index = month_ends.index

# A missing month-end makes this month's and next month's returns NaN
monthly_returns = ri_month_end.pct_change(fill_method=None)
monthly_returns = monthly_returns.iloc[1:]                         # first month: no previous month-end
monthly_returns = monthly_returns.drop(index=incomplete_final.index)

n_valid = monthly_returns.dropna().shape[0]
print(f"Monthly returns: {len(monthly_returns)} months ({n_valid} available), "
      f"from {monthly_returns.index.min()} to {monthly_returns.index.max()}")

add_check("Incomplete final month", "All", list(incomplete_final), len(incomplete_final),
          excluded=len(incomplete_final),
          justification="No case" if len(incomplete_final) == 0
          else "Data end before month-end; monthly return not computed")
add_check("Missing month-end", "All", list(missing_month_end), len(missing_month_end),
          excluded=len(missing_month_end),
          justification="No case" if len(missing_month_end) == 0
          else "Month-end absent from data; returns of this month and the next set to NaN")

# ---------------------------------------------------------------------------
# Consistency test: monthly return = compounded daily returns, fully covered months only
# ---------------------------------------------------------------------------
sessions_per_month = full_sessions.to_series().groupby(full_sessions.to_period("M")).size()
data_per_month = (ri_clean.index.to_series().groupby(ri_clean.index.to_period("M")).size()
                  .reindex(sessions_per_month.index))
full_months = sessions_per_month.index[sessions_per_month.eq(data_per_month)]

compounded = (1 + r_ri.iloc[1:]).groupby(r_ri.index[1:].to_period("M")).prod() - 1
common = (monthly_returns.dropna().index
          .intersection(compounded.index)
          .intersection(full_months))
max_diff = (monthly_returns.loc[common] - compounded.loc[common]).abs().max().max()

print(f"Months tested: {len(common)}, max difference: {max_diff:.2e}")
assert max_diff < 1e-10, "Monthly returns inconsistent with daily returns"

# ---------------------------------------------------------------------------
# Risk-free rate audit (DGS1MO, FRED): no transformation yet
# ---------------------------------------------------------------------------
rf = pd.read_csv(RF_FILE)
rf.columns = ["Date", "DGS1MO"]
rf["Date"] = pd.to_datetime(rf["Date"], errors="coerce")
rf["DGS1MO"] = pd.to_numeric(rf["DGS1MO"], errors="coerce")   # blanks become NaN
rf = rf.dropna(subset=["Date"]).set_index("Date").sort_index()["DGS1MO"]

print(f"DGS1MO: {len(rf)} dates, {rf.index.min().date()} to {rf.index.max().date()}")
print(f"Duplicated dates: {rf.index.duplicated().sum()}")
print(f"Missing values: {rf.isna().sum()}")

# Missing values on NYSE sessions (e.g. bond market holidays such as Columbus Day)
rf_on_sessions = rf.reindex(sessions)
missing_on_sessions = rf_on_sessions.index[rf_on_sessions.isna()]
print(f"NYSE sessions without a published rate: {len(missing_on_sessions)}")
print("Examples:", [str(d.date()) for d in missing_on_sessions[:8]])

# Longest gap between two published rates on NYSE sessions (staleness)
published = rf_on_sessions.notna()
gap_len = (~published).groupby(published.cumsum()).sum().max()
print(f"Longest run of NYSE sessions without a published rate: {gap_len}")

print(f"Range: {rf.min():.2f}% to {rf.max():.2f}%  (negative values: {(rf < 0).sum()})")

add_check("Risk-free rate missing on NYSE session", "DGS1MO", missing_on_sessions,
          len(missing_on_sessions),
          justification="To be filled with last published rate (V1.4 rule); transformation postponed")

pd.DataFrame(report).to_csv(REPORTS / "data_quality_report.csv", index=False)

,Mean annual gap RI - Price,Days with gap > +0.1%,Days with gap < -0.1%,Largest daily gap
SPY,2.06%,79,0,0.806%
EFA,3.14%,39,0,2.496%
EEM,2.20%,38,0,2.091%
IEF,2.47%,216,0,0.835%
LQD,3.98%,238,0,0.554%
TLT,3.11%,238,0,0.440%
VNQ,4.32%,79,0,2.647%
GLD,0.00%,0,0,0.007%


Daily returns: 4991, irregular: 0
Month-ends: 239
Incomplete final month: ['2026-11']
Missing month-ends (data gap): []
Monthly returns: 237 months (237 available), from 2007-02 to 2026-10
Months tested: 237, max difference: 1.33e-15
DGS1MO: 5176 dates, 2007-01-02 to 2026-11-03
Duplicated dates: 0
Missing values: 212
NYSE sessions without a published rate: 38
Examples: ['2007-10-08', '2007-11-12', '2008-10-13', '2008-11-11', '2009-10-12', '2009-11-11', '2010-10-11', '2010-11-11']
Longest run of NYSE sessions without a published rate: 1
Range: 0.00% to 6.02%  (negative values: 0)


In [6]:
from pathlib import Path

PROCESSED = Path("../data/processed")
REPORTS = Path("../data/reports")
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# 1. Clean daily prices and total return indices (NYSE sessions only)
daily_clean = pd.concat(
    [price_clean.add_prefix("P_"), ri_clean.add_prefix("RI_")], axis=1
)
daily_clean.index.name = "Date"
daily_clean.to_csv(PROCESSED / "etf_daily_clean.csv")

# 2. Daily returns from RI, with regularity flag
daily_returns.index.name = "Date"
daily_returns.to_csv(PROCESSED / "etf_daily_returns.csv")

# 3. Monthly returns, with the month-end date used for each month
monthly_out = monthly_returns.copy()
monthly_out.insert(0, "MonthEnd", month_ends.loc[monthly_out.index].dt.date.values)
monthly_out.index = monthly_out.index.astype(str)
monthly_out.index.name = "Month"
monthly_out.to_csv(PROCESSED / "etf_monthly_returns.csv")

# 4. Data quality report
report_df = pd.DataFrame(report)
report_df.to_csv(REPORTS / "data_quality_report.csv", index=False)

for f in [PROCESSED / "etf_daily_clean.csv", PROCESSED / "etf_daily_returns.csv",
          PROCESSED / "etf_monthly_returns.csv", REPORTS / "data_quality_report.csv"]:
    print(f"{f.name}: {pd.read_csv(f).shape[0]} rows")

etf_daily_clean.csv: 4992 rows
etf_daily_returns.csv: 4991 rows
etf_monthly_returns.csv: 237 rows
data_quality_report.csv: 37 rows


In [7]:
# Check that src/data.py reproduces the notebook results
import sys
import importlib
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # project root, so "src" can be imported
import src.data as data
importlib.reload(data)  # force Python to re-read the file from disk

print("File loaded:", data.__file__)
print("Has build_clean_dataset:", hasattr(data, "build_clean_dataset"))
# Check that src/data.py reproduces the notebook results
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # project root, so "src" can be imported
from src import data

out = data.build_clean_dataset()

print("Sessions kept:     ", len(out["ri"]))
print("Excluded dates:    ", len(out["excluded_dates"]))
print("Daily returns:     ", len(out["daily_returns"]))
print("Irregular returns: ", int((~out["daily_returns"]["regular"]).sum()))
print("Monthly returns:   ", len(out["monthly_returns"]))
print("Missing month-ends:", len(out["missing_month_end"]))
print("Incomplete final:  ", list(out["incomplete_final"].index.astype(str)))

# Compare with the files exported by the notebook
clean, daily, monthly = data.load_processed()
cols = data.TICKERS
print("Max diff daily:  ", float((out["daily_returns"][cols] - daily[cols]).abs().max().max()))
print("Max diff monthly:", float((out["monthly_returns"][cols] - monthly[cols]).abs().max().max()))



# 12-1 momentum: return from month-end t-12 to month-end t-1 (Total Return Index)

import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # project root, so "src" can be imported
from src import data

out = data.build_clean_dataset()
monthly = out["monthly_returns"]
month_ends = out["month_ends"]
ri = out["ri"]

# RI at each NYSE month-end (NaN if that session is missing), incomplete final month removed
ri_me = ri.reindex(month_ends.values)
ri_me.index = month_ends.index
ri_me = ri_me.drop(index=out["incomplete_final"].index)

# Signal at month-end t uses only RI up to month-end t-1: no look-ahead
momentum = ri_me.shift(1) / ri_me.shift(12) - 1

# Consistency check: same value by compounding monthly returns t-11 ... t-1
check = (1 + monthly).shift(1).rolling(11, min_periods=11).apply(np.prod, raw=True) - 1
common = momentum.dropna().index.intersection(check.dropna().index)
max_diff = float((momentum.loc[common] - check.loc[common]).abs().max().max())

assert len(common) > 0, "No valid months for momentum comparison"
assert max_diff < 1e-10, "Momentum calculation inconsistent"

# Timing convention: momentum[t] is observed at the close of month t
# (uses RI up to month-end t-1, month t is skipped); weights decided at t
# are traded on the next session and earn the returns of month t+1.

print("First available signal:", momentum.dropna().index[0])
print("Last available signal: ", momentum.dropna().index[-1])
print("Months with a signal:  ", len(momentum.dropna()))
print("Max diff vs compounding:", max_diff)
print("\nLatest 12-1 momentum (%):")
print((momentum.iloc[-1] * 100).round(2).sort_values(ascending=False))

# Export the 12-1 momentum signal (decimal returns, one row per decision month-end)
export = momentum.dropna(how="all").copy()
export.index = export.index.astype(str)
export.index.name = "Month"
export.insert(0, "MonthEnd", month_ends.loc[momentum.dropna(how="all").index].dt.strftime("%Y-%m-%d").values)

path = data.PROCESSED_DIR / "momentum_12_1.csv"
export.to_csv(path)

print("Saved:", path.name, export.shape)
print(export.head(3))

File loaded: /Users/abrahamadegoke/Desktop/Quantitative asset /Project/src/data.py
Has build_clean_dataset: True
Sessions kept:      4992
Excluded dates:     0
Daily returns:      4991
Irregular returns:  0
Monthly returns:    237
Missing month-ends: 0
Incomplete final:   ['2026-11']
Max diff daily:   0.0
Max diff monthly: 0.0
First available signal: 2008-01
Last available signal:  2026-10
Months with a signal:   226
Max diff vs compounding: 8.881784197001252e-16

Latest 12-1 momentum (%):
EEM    23.13
EFA    13.71
SPY    13.02
VNQ     4.32
GLD     3.46
LQD    -4.17
IEF    -4.42
TLT   -10.21
Name: 2026-10, dtype: float64
Saved: momentum_12_1.csv (226, 9)
           MonthEnd       SPY       EFA       EEM       IEF       LQD  \
Month                                                                   
2008-01  2008-01-31  0.035764  0.084301  0.331531  0.108126  0.036341   
2008-02  2008-02-29 -0.007370  0.000000  0.263005  0.122341  0.042280   
2008-03  2008-03-31 -0.044121 -0.037198  0.21